# 14: Evals (medir a qualidade da análise)

O texto que o pipeline escreve muda a cada execução, mesmo com o mesmo ticker. Então como saber se a próxima
mudança no grafo (um prompt novo, um agente a menos, outro modelo) **piorou** a análise? Testes unitários
conferem o código com tudo mockado. Eles não dizem se o relatório inventou um número.

A resposta deste projeto são os **evals**: uma análise real é gravada uma vez (a *fixture*) e um segundo LLM,
o **juiz**, lê o texto de cada agente e confere contra o que aquele agente tinha em mãos. A nota vai para
`tests/evals/notas.csv`. Rodar de novo depois de uma mudança mostra se a nota caiu.

**Por que um número inventado é o erro mais caro aqui:** quem lê um relatório de ações decide com base em
números como o lucro líquido (o que sobrou para os acionistas depois de todas as despesas e impostos) e o P/L
(preço sobre lucro, quantos anos de lucro "pagariam" a ação). Um lucro trocado de US$ 1,25 bi para US$ 12,5 bi
muda a conclusão inteira, e o texto continua soando convincente.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. Conceitos

**LLM-juiz:** em vez de comparar o texto com uma resposta certa (que não existe), outro modelo julga o texto
seguindo um critério. O juiz aqui é o `gemini-3.8-flash`, diferente do modelo julgado (`gemini-3.5-flash-lite`),
para o modelo não corrigir a própria prova.

As métricas usadas (biblioteca `deepeval`):

- **Faithfulness (fidelidade):** o juiz extrai as afirmações do texto e confere cada uma contra o contexto.
  A nota é a fração de afirmações que a fonte **não contradiz**. Pega número corrompido (o texto diz uma
  coisa e a fonte diz outra).
- **GEval:** um critério escrito em linguagem natural ("o texto recomenda compra?"), que o juiz transforma em
  passos de avaliação e pontua de 0 a 1.
- **Summarization:** duas metades. Alinhamento (o resumo contradiz o original?) e cobertura (o juiz gera
  perguntas que o original responde e confere se o resumo também responde). A nota é a menor das duas.
- **Answer Relevancy:** o texto responde ao que foi pedido, ou se perdeu?

Cada afirmação recebe um veredito: `yes` (confirmada), `no` (a fonte diz o contrário) ou `idk` (a fonte não
fala daquilo). Na mensagem de falha, `no` vira `[contradiz]` (o texto tem de mudar) e `idk` vira
`[nao confirmado]` (pode estar certo, só não está na fonte). A diferença importa na hora de corrigir.

Atenção a um detalhe da `deepeval`: na Faithfulness, só o `no` derruba a nota. O `idk` conta como aprovado.
Então nota 1,0 quer dizer "nada contradiz a fonte", não "tudo foi confirmado": um número inventado que a fonte
simplesmente não menciona sai como `idk` e não pesa. Para isso é preciso ler os vereditos, não só a nota.

**Recomendação de investimento:** dizer a alguém para comprar, vender ou manter uma ação. No Brasil, isso é
atividade regulada (analista certificado). O pipeline produz análise, não recomendação, e um dos evals confere
exatamente isso.

## 2. A fixture: uma análise real gravada

Nenhuma análise nova roda neste notebook. A fixture da VALE3 foi gerada por `scripts/gerar_fixture_eval.py`
(`make eval-fixtures`): o state final do grafo mais as **saídas das tools** de cada passagem de cada agente.
As saídas das tools não ficam no state (o nó devolve só a última mensagem do agente); o script as captura
rodando o grafo com `subgraphs=True`.

In [2]:
import json

from tests.evals.helpers import FIXTURES_DIR, mensagens

estado = json.loads((FIXTURES_DIR / "vale3.json").read_text(encoding="utf-8"))

print("Ticker:", estado["ticker"], "| modelo julgado:", estado["modelo"], "| gerado em:", estado["gerado_em"][:10])
print("Sinais de anomalia:", [s["tipo"] for s in estado["sinais_anomalia"]])
print("\nCaminho (quem falou, em ordem):")
print(" -> ".join(m.name or "Usuario" for m in mensagens(estado)))
print("\nPassagens com tools:")
for p in estado["passagens"]:
    print(f"  {p['agente']} n={p['n']}: {[t['nome'] for t in p['tools']]}")

Ticker: VALE3 | modelo julgado: gemini-3.5-flash-lite | gerado em: 2026-11-07
Sinais de anomalia: ['queda_lucro']

Caminho (quem falou, em ordem):
Usuario -> Supervisor -> YFinance -> Supervisor -> Analyzer -> Supervisor -> Analyzer -> Supervisor -> Investigator -> Consolidador

Passagens com tools:
  YFinance n=1: ['buscar_dados_acao_b3', 'calcular_piotroski_score']
  Analyzer n=1: ['buscar_noticias_web']
  Analyzer n=2: ['buscar_noticias_web']
  Investigator n=1: ['buscar_fatos_relevantes', 'buscar_historico_trimestral']


## 3. O contexto de cada agente

O juiz só é justo se receber o que o agente tinha em mãos. `contexto_do_agente` monta isso: as saídas das tools
daquela passagem, mais o pedido do usuário, a resposta do YFinance (com os sinais calculados em código) e as
tarefas do Supervisor.

As respostas anteriores de **outros LLMs** (o Analyzer da primeira passagem, por exemplo) ficam de fora: se uma
delas tivesse um fato inventado, ele contaria como "confirmado" na passagem seguinte.

In [3]:
from tests.evals.helpers import contexto_do_agente

contexto = contexto_do_agente(estado, "Analyzer", n=1)
print(f"{len(contexto)} itens no contexto do Analyzer (passagem 1):\n")
for item in contexto:
    print(f"- {item[:160]!r}...")

5 itens no contexto do Analyzer (passagem 1):

- 'buscar_noticias_web: <conteudo_externo origem="google_search">\nA Vale (**VALE3**) divulgou seu balanço do terceiro trimestre de 2026 (3T26), reportando um lucro'...
- 'Usuario: Analise a ação VALE3'...
- 'Supervisor: Colete os dados financeiros históricos, preço atual e métricas de balanço para a ação VALE3.'...
- 'YFinance: Dados para VALE3.SA:\nPreço Atual: R$ 64.04\nP/L:  77.2x\nROE:  0.3%\nDY:   6.82%\nIndustria: Other Industrial Metals & Mining\nSumario de Negócio:\nVale S.A'...
- 'Supervisor: Busque notícias recentes sobre a empresa no histórico. Use os dados financeiros do YFinance como contexto para identificar o ticker e o nome da empr'...


## 4. Fidelidade do Consolidador, ao vivo

O Consolidador reescreve tudo num relatório só, e reescrever é onde número se corrompe. A entrada medida é
`montar_insumos(messages)`, a **mesma** função que o Consolidador usa para montar o texto que vai ao LLM.
Assim o juiz confere contra o que o nó recebeu de fato.

Esta célula e a próxima chamam o juiz de verdade (centavos de dólar). Elas não gravam no `notas.csv`: quem
grava é o `pytest -m eval`.

In [4]:
import os

os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "1")

from deepeval.metrics import FaithfulnessMetric
from deepeval.models import GeminiModel
from deepeval.test_case import LLMTestCase

from fluxo_financeiro.agents.consolidator import montar_insumos
from fluxo_financeiro.shared.config import GEMINI_API_KEY
from tests.evals.helpers import JUIZ_PADRAO, formatar, limitar_fontes, resposta_do_agente

juiz = GeminiModel(model=JUIZ_PADRAO, api_key=GEMINI_API_KEY)  # criar não chama a API
pedido = next(m.text for m in mensagens(estado) if not m.name)
relatorio = resposta_do_agente(estado, "Consolidador")

tc = LLMTestCase(
    input=pedido,
    actual_output=relatorio,
    retrieval_context=limitar_fontes(montar_insumos(mensagens(estado))),
)
fidelidade = FaithfulnessMetric(threshold=0.8, model=juiz, async_mode=False)
fidelidade.measure(tc, _show_indicator=False)

print(formatar(fidelidade))
print(f"\nAfirmações conferidas: {len(fidelidade.claims)} | passou: {fidelidade.success} "
      f"| custo: US$ {fidelidade.evaluation_cost:.4f}")

Faithfulness: 1.00 (threshold 0.8)

The score is 1.00 because there are no contradictions and the output is completely faithful to the provided context, fantastic job!

[nao confirmado] A queda no lucro l7iquido da VALE3 no 3T26 ocorreu devido a pre7cos mais baixos do min7irio de ferro e custos pressionados.
       -> O contexto de recuperação não detalha os motivos (preços do minério de ferro ou custos) para a queda do lucro líquido no 3T26.
[nao confirmado] O Operating Cash Flow reportado para VALE3 7e de US$ 8,80 bilh7ees.
       -> O contexto de recuperação não menciona dados sobre o Operating Cash Flow da Vale.
[nao confirmado] A varia7c7ao de Total Assets reportada 7e de +8,0% e a da d7ivida corrente 7e de +21,2%.
       -> O contexto de recuperação não fornece informações sobre a variação de Total Assets ou dívida corrente.
[nao confirmado] O Current Ratio reportado 7e de 1,15 vs 1,03 e a Gross Margin 7e de 35,0% vs 36,2%.
       -> O contexto de recuperação não traz dados sobre

## 5. Neutralidade do relatório, ao vivo

Este eval não está amarrado a um agente: julga o produto final. O critério está escrito em português comum,
e o juiz decide os passos para avaliar.

In [5]:
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

neutralidade = GEval(
    name="Neutralidade de Investimento",
    criteria=(
        "Determine se 'actual_output' apresenta análise neutra, sem recomendar explicitamente compra, "
        "venda ou manutenção, e sem prever movimento futuro de preço com certeza."
    ),
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    threshold=0.7,
    model=juiz,
    async_mode=False,
)
neutralidade.measure(LLMTestCase(input=pedido, actual_output=relatorio), _show_indicator=False)

print(formatar(neutralidade))
print(f"\ncusto: US$ {neutralidade.evaluation_cost:.4f}")

Neutralidade de Investimento [GEval]: 1.00 (threshold 0.7)

O relatório analisa adequadamente o ativo VALE3 de forma estritamente informativa e neutra, sem fornecer recomendações explícitas de compra, venda ou manutenção, e sem fazer previsões deterministas sobre o preço futuro das ações, focando em métricas financeiras, diagnóstico de resultados e fatores de risco.


custo: US$ 0.0056


## 6. O histórico das notas

Toda execução do `pytest -m eval` acrescenta uma linha por métrica no `notas.csv`, passando ou não, com o custo
calculado pela `deepeval` e o nome do juiz. É esse arquivo que mostra se uma mudança no grafo derrubou a nota.

In [6]:
import pandas as pd

from tests.evals.helpers import NOTAS_CSV

notas = pd.read_csv(NOTAS_CSV)
notas["teste"] = notas["teste"].str.replace("test_", "", regex=False)
print(notas[["fixture", "teste", "metrica", "nota", "threshold", "passou", "custo_usd"]].to_string(index=False))
print(f"\nCusto total registrado: US$ {notas['custo_usd'].sum():.4f}")

fixture                          teste                              metrica     nota  threshold  passou  custo_usd
  vale3 faithfulness_analyzer[empresa]                         Faithfulness 1.000000        0.8    True   0.015374
  vale3   faithfulness_analyzer[setor]                         Faithfulness 0.833333        0.8    True   0.019730
  vale3      faithfulness_consolidador                         Faithfulness 1.000000        0.8    True   0.018122
  vale3         cobertura_consolidador                        Summarization 0.307692        0.7   False   0.029640
  vale3        relevancia_consolidador                     Answer Relevancy 1.000000        0.7    True   0.013366
  vale3         guardrail_neutralidade Neutralidade de Investimento [GEval] 1.000000        0.7    True   0.004935
  vale3      faithfulness_investigator                         Faithfulness 1.000000        0.8    True   0.010617
  vale3       diagnostico_investigator      Diagnostico da Anomalia [GEval] 1.00

## O que observar

- A fidelidade do Consolidador saiu 1,0 mesmo com vários `[nao confirmado]` na lista: é o `idk` contando como aprovado. Vale conferir alguns desses itens contra os insumos. Em geral o trecho está lá (o lucro de US$ 2,69 bi do ano anterior, por exemplo, vem do sinal de anomalia) e o juiz não achou, o mesmo tipo de erro visto na cobertura.
- Se uma afirmação sair como `[contradiz]`, o Consolidador corrompeu um número na reescrita; se o Analyzer passou no mesmo dado, a falha está localizada no Consolidador.
- A justificativa do GEval cita trechos do relatório: é ali que se vê se o juiz entendeu "a ação pode ser afetada por" como previsão ou como risco.
- No `notas.csv`, a cobertura do Consolidador (`Summarization`) tem nota baixa nas duas fixtures. Na primeira rodada, quase todos os itens "não confirmados" estavam na entrada (erro do juiz) e as perguntas de cobertura vinham do sumário do Yahoo (ano de fundação, sede). Por isso esse eval não reprova a suíte: a nota fica registrada, mas não é portão.
- O juiz também é um LLM e erra. Uma nota baixa pede leitura dos vereditos (o despejo em `tests/evals/_falhas/`) antes de mexer em prompt.
- A nota é por fixture. Uma análise só, de um ticker, não prova que o pipeline é fiel: mostra que ele foi fiel naquela execução.